# Парсинг данных с сайта на Python

## Подключение библиотек

In [1]:
from bs4 import BeautifulSoup as bs
import requests
import pandas as pd

## Получение информаций

In [2]:
base_url = 'https://premier.one'
sub_url = '/collections/top-250-filmov-na-premier?page='

In [3]:
result_list = {'title': [], 'genre': [], 'year': [],
               'country': [], 'rate': [], 'description': []}

In [4]:
count_films = 0
for i in range(1, 6):
    url = base_url + sub_url + str(i)
    page = requests.get(url)
    page.status_code
    soup = bs(page.text, 'html.parser')
    links = soup.find_all('a', class_="e-poster")
    for link in links:
        try:
            url = base_url + link.get('href')
            page = requests.get(url)
            inner_soup = bs(page.text, 'html.parser')

            t_el = inner_soup.find('div', class_='w-show-promo-new__title-small')
            y_el = inner_soup.find('a', class_='w-show-promo-new__year-link')
            c_el = inner_soup.find('a', class_='w-show-promo-new__country-link')
            r_el = inner_soup.find('span', class_='a-badge-rating__value')
            
            d_v = inner_soup.find('span', class_='a-cropped-text__visible-text')
            d_h = inner_soup.find('span', class_='a-cropped-text__hide')
            full_desc = (d_v.text.strip() if d_v else "") + (d_h.text.strip() if d_h else "")
    
            result_list['title'].append(t_el.text.strip() if t_el else "Без названия")
            result_list['year'].append(y_el.text.strip() if y_el else "—")
            result_list['country'].append(c_el.text.strip() if c_el else "—")
            result_list['rate'].append(r_el.text.strip() if r_el else "0.0")
            result_list['description'].append(full_desc if full_desc else "Нет описания")
            
            g_els = inner_soup.find_all('a', class_='w-show-promo-new__genre-link')
            result_list['genre'].append(", ".join([i.text.strip() for i in g_els]) if g_els else "—")
            
            count_films += 1
    
        except Exception as e:
            print(f"Ошибка на {url}: {e}")
            continue


In [5]:
count_films

250

In [6]:
print("Количество нулевых значений в: ")
for i in result_list:
    print( i + " - " + str(result_list[i].count(None)))

Количество нулевых значений в: 
title - 0
genre - 0
year - 0
country - 0
rate - 0
description - 0


In [7]:
for key, value in result_list.items():
    print(f"{key}: {len(value)}")

title: 250
genre: 250
year: 250
country: 250
rate: 250
description: 250


In [8]:
file_name = 'Top250FilmsInPremier.csv'
df = pd.DataFrame(data=result_list)
df.to_csv(file_name)
df.head(15)

,title,genre,year,country,rate,description
0,Чебурашка 2,"Фэнтези, Комедия, Приключения, Семейный",2026,Россия,8.4,Уже год Чебурашка живет у Гены. Шаловливый уша...
1,Несвятая Валентина,"Комедия, Мелодрама",2026,Россия,7.4,Зимняя романтическая комедия с Анастасией Крас...
2,Добрый доктор,Комедия,2026,Россия,8.5,Комедия от создателей «Полицейский с Рублевки....
3,Праведник,"Драма, История, Военный",2023,Россия,8.9,"Александр Яценко, Константин Хабенский и Марк ..."
4,Мистер Нокаут,"Драма, Спорт, Биография",2022,Россия,8.9,"Драматичный путь к славе советского атлета, ст..."
5,На острие,"Драма, Спорт",2019,Россия,9.0,Александра Покровская долго и упорно поднимала...
6,Алиса в Стране Чудес (2025),"Фэнтези, Приключения, Мюзикл",2025,Россия,7.1,Яркое музыкальное приключение по стихам Владим...
7,Культурная комедия,Комедия,2024,Россия,8.4,Комедия с Алексеем Чадовым и Юрием Стояновым о...
8,Первый на Олимпе,"Драма, Спорт, Биография",2025,Россия,9.0,История завоевания первого олимпийского золота...
9,"Двое в одной жизни, не считая собаки",Мелодрама,2025,Россия,8.7,Жизненная и добрая картина получила приз зрите...


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        250 non-null    object
 1   genre        250 non-null    object
 2   year         250 non-null    object
 3   country      250 non-null    object
 4   rate         250 non-null    object
 5   description  250 non-null    object
dtypes: object(6)
memory usage: 11.8+ KB
